# CIT: feature extractor evaluation

**Not run.** The repo releases no trained CIT weights and is missing code needed to build or run the model, so there is no env and nothing below section 2 has been executed. The model card and notes come from reading the repo at the commit below.

## Notes and verdict

**Hardware used for the speed numbers:** n/a (not run)

**Status:** inspected only; no env, no weights, not run on `test_pictures`.

**Strengths:**

**Weaknesses:**
- **No trained weights.** The only download (`models/pretrained/Backbone.txt`, OneDrive) is the ImageNet-pretrained HRNet backbone, not the trained CIT model. No GitHub releases. Issue #1 (Sept 2025, open, no reply) asks for the full checkpoint.
- **Incomplete code.** `models/core/model.py` imports `models/core/GCN_base.py`, which isn't in the repo; there is no class that assembles the full model (only the landmark, occlusion, and pose decoders); no config files; no train, test, or demo script. Issue #1 lists the same gaps. `models/core/xx` is a 1-byte placeholder file.
- **No requirements file.** Imports include torch, torchvision, timm, einops, opencv, scipy, hdf5storage, PIL, matplotlib.
- Training data (MERL-RAV-FLOP) is only available by emailing the authors.

**Verdict:** drop (cannot evaluate)

**Why:** Without trained weights and with parts of the model code missing, it can't be run on our clips like the other models. Revisit only if the authors release the full checkpoint and the missing files.

## Setup

Imports and paths. Run this first.

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()

from src import clip_io, plots, schema

MODEL = 'CIT'
MODEL_DIR = ROOT / 'models' / MODEL

## 1. Model card

| Field | Value |
|---|---|
| Paper | Li, Tan, Gou. *Cascaded Iterative Transformer for Jointly Predicting Facial Landmark, Occlusion Probability and Head Pose.* IJCV 2023, https://link.springer.com/article/10.1007/s11263-023-01935-2 |
| Repo URL | https://github.com/Iron-LYK/CIT |
| Commit hash | `f126f875c2fbcc7bad280506cc4013cceea1e230` (2025-12-10) |
| License | MIT (LICENSE file) |
| Weights | **None released** for the full model. Only an ImageNet-pretrained HRNet backbone (OneDrive link in `models/pretrained/Backbone.txt`) |
| Pipeline | HRNet backbone → shared transformer encoder → cascaded, iterative decoders for landmarks, occlusion probability, and pose, each passing its predictions to the others (paper; assembly code not in the repo) |
| Inputs required | Face crop; size and normalization not documented (no config files) |
| Outputs | 68 facial landmarks, per-landmark occlusion probability, head pose. No gaze, no AUs |
| Native units / conventions | Pose predicted as a 6D rotation; `utils.py` converts it to a rotation matrix and Euler angles (radians). Axis order and signs not checked |
| Conversion to schema | Not done (not run) |
| Env | None created |

In [ ]:
!git -C {MODEL_DIR} remote get-url origin
!git -C {MODEL_DIR} rev-parse HEAD

## 2. Setup log

Not set up: the repo has no trained weights and is missing code (see notes).

| # | Step / command | Result | Error | Fix |
|---|---|---|---|---|
| 1 | Inspect repo (README, `models/`, imports, GitHub releases and issues) | No full checkpoint, no inference script, `GCN_base.py` missing, no configs, no requirements file | | Stopped before creating an env |

## 3. Run on test clips

**Not run** (no weights). Cells below are the unchanged template, except `CATEGORIES`.

`predict(image)` is the only model-specific code. It takes one BGR frame and returns `(face_conf, outputs)`: `outputs` is a dict of schema column names to values, already converted to schema units and signs, or `None` when no face is found. List every column it can return in `OUTPUT_COLUMNS`.

In [ ]:
# TODO: load the model
model = None

In [ ]:
# TODO: every column predict() can return, in a fixed order.
# Frames with no face get NaN in all of them, so clips with no detections still save.
OUTPUT_COLUMNS = []

def predict(image):
    """TODO: run the model on one BGR frame.

    Returns (face_conf, outputs). outputs is e.g.
    {'AU12_int': 1.3, 'gaze_yaw_deg': -4.2, ...} or None if no face.
    """
    raise NotImplementedError

In [ ]:
CATEGORIES = ['head_pose']
TEST_SET = 'test_pictures'  # folder under data/ with manifest.csv
manifest = clip_io.load_manifest(TEST_SET, CATEGORIES)
manifest

In [ ]:
results, timings = {}, {}
for row in manifest.itertuples():
    records = []
    t0 = time.perf_counter()
    for frame, timestamp_ms, image in clip_io.iter_frames(clip_io.clip_path(row)):
        face_conf, outputs = predict(image)
        record = {'clip_id': row.clip_id, 'frame': frame, 'timestamp_ms': timestamp_ms,
                  'face_conf': float(face_conf) if outputs is not None else 0.0}
        record.update(outputs or {})
        records.append(record)
    timings[row.clip_id] = time.perf_counter() - t0
    results[row.clip_id] = pd.DataFrame.from_records(records, columns=list(schema.REQUIRED_COLUMNS) + OUTPUT_COLUMNS)
    print(f'{row.clip_id}: {len(records)} frames in {timings[row.clip_id]:.1f} s')

## 4. Save output in the common schema

`save_output` runs `schema.validate` and refuses to write a file that fails it. Fix the conversion in `predict`, not the check.

In [ ]:
for clip_id, df in results.items():
    df = df.astype({'frame': 'int64', 'timestamp_ms': 'float64', 'face_conf': 'float64'})
    out_cols = [c for c in df.columns if c not in schema.REQUIRED_COLUMNS and not c.startswith(schema.EXTRA_PREFIX)]
    df[out_cols] = df[out_cols].astype('float64')
    df = schema.add_missing_au_pairs(df)
    print(clip_io.save_output(df, MODEL, clip_id))

## 5. Quick checks

These read back the saved files, so they test exactly what was written.

In [ ]:
out = clip_io.load_outputs(MODEL, manifest['clip_id'])  # only this run's clips, not leftovers from other runs
print('output columns:', schema.output_columns(out))

### Face-detection rate

In [ ]:
display(plots.detection_rate(out).to_frame())
plots.plot_detection_rate(out);

### Speed (fps on this machine)

Includes video decoding. Note the hardware (CPU/GPU) in the verdict.

In [ ]:
plots.fps_table(timings, out)

### Standard plots per clip

In [ ]:
for clip_id, df in out.groupby('clip_id'):
    plots.plot_face_conf(df)
    if schema.au_ids(df):
        for kind in ('int', 'occ'):
            if df.filter(like=f'_{kind}').notna().any().any():
                plots.plot_aus(df, kind)
    if any(c in df for c in schema.GAZE_COLUMNS):
        plots.plot_gaze(df)
    if any(c in df for c in schema.POSE_COLUMNS):
        plots.plot_pose(df)
    plt.show()

### Plausibility and stability (no ground truth)

- Value ranges: do angles and AU levels look reasonable for what happens in each clip?
- Jitter: median absolute frame-to-frame change on detected frames. A high value on a still face means a noisy model.
- AU occurrence rate: AUs that are always on or always off are suspicious.

In [ ]:
detected = out[out['face_conf'] >= plots.DEFAULT_CONF_THRESHOLD]
display(detected[schema.output_columns(out)].describe().T)

angle_cols = [c for c in schema.GAZE_COLUMNS + schema.POSE_COLUMNS if c in out]
if angle_cols:
    jitter = detected.groupby('clip_id')[angle_cols].apply(lambda d: d.diff().abs().median())
    display(jitter.rename(columns=lambda c: c.replace('_deg', '_jitter_deg')))

occ_cols = [c for c in out if c.endswith('_occ') and out[c].notna().any()]
if occ_cols:
    display(detected.groupby('clip_id')[occ_cols].mean())

### Failure cases

Frames where the face was missed or detected with low confidence. Look at a few to see why (pose, occlusion, lighting, blur).

In [ ]:
failures = plots.failure_frames(out)
display(failures.groupby('clip_id').size().rename('n_failures').to_frame())
failures.head(20)

In [ ]:
def show_frames(clip_id, frames, ncols=4):
    row = manifest.set_index('clip_id').loc[clip_id]
    wanted = set(frames)
    images = [(f, img) for f, _, img in clip_io.iter_frames(clip_io.clip_path(row), rgb=True) if f in wanted]
    if not images:
        return
    nrows = -(-len(images) // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3 * ncols, 2.5 * nrows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, (f, img) in zip(axes.flat, images):
        ax.imshow(img)
        ax.set_title(f'{clip_id} frame {f}', fontsize='small')
    plt.show()

# Up to 8 evenly spaced failure frames per clip
for clip_id, f in failures.groupby('clip_id'):
    show_frames(clip_id, f['frame'].iloc[np.linspace(0, len(f) - 1, min(8, len(f))).astype(int)])